# BTS Digital Twin (NVS) — Compact Gaussian, private set1: train 1 scene / lần chạy

Notebook này train **Compact Gaussian** (Lee et al., arXiv:2311.13681, mục 3.1 —
"Gaussian Volume Mask") + bảo vệ vùng chi tiết cao (ăng-ten/RRU/cáp) tuỳ chọn, thay cho
3DGS vanilla — xem `pipeline/extra/compact_gaussian.py` + `train_compact.py`. Chỉ xử lý
**1 scene private duy nhất mỗi lần chạy** (biến `SCENE` ở
Bước 6). Có **8 scene private**: `HCM0249`, `HCM0254`, `HCM0276`, `HCM1439`, `HNI0131`, `HNI0265`, `HNI0366`, `HNI0437`.

Cách dùng: giống hệt notebook public — đổi `SCENE` ở Bước 6 rồi Save Version, lặp lại
cho 8 scene (có thể chạy song song 2 version). Mỗi scene độc lập hoàn toàn (không có
"nền tảng" chung giữa các scene, cũng không liên quan gì tới các scene public).

**Khác với notebook public: KHÔNG có bước tính điểm PSNR/SSIM** — `private_set1` không
có ảnh ground-truth (BTC giữ lại để tự chấm), nên Bước 6 chỉ render rồi hiển thị vài
ảnh để tự mắt kiểm tra hợp lý (không bị nhiễu/méo/sai màu bất thường), không ra điểm số.

**Trước khi chạy, cần điền:**
1. Settings → Accelerator: **GPU T4 x2** (hoặc P100) → Internet: **On**.
2. `REPO_URL` ở Bước 3, `GDRIVE_URL` ở Bước 4 (đã điền sẵn), `SCENE` ở Bước 6.
3. 1 scene private mất khoảng **~2.5-3 giờ** (30000 iterations) — 1 session thừa sức xong 1 scene.

**Bảo mật:** để notebook này **Private**.

## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)

In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting (pin commit tương thích Compact Gaussian)

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Bước build
2 CUDA extension (`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.

**Pin commit `54c035f7834b564019656c3e3fcc3646292f727d`**: `pipeline/extra/compact_gaussian.py`
override trực tiếp các hàm nội bộ của `scene/gaussian_model.py` (`densification_postfix`,
`prune_points`, cơ chế `tmp_radii` trong `densify_and_prune`...) — đã đối chiếu đúng
signature các hàm này tại chính commit trên, nên PHẢI pin đúng commit đó thay vì dùng
`main` mới nhất (dễ vỡ âm thầm nếu upstream đổi signature sau này). Cũng là yêu cầu tái
lập ở đề bài mục 10.3.

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
%cd /kaggle/working
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Nếu push CẢ project (gồm `Đề bài.md`, `KE_HOACH_VONG1.md`, `Dataset/`, `pipeline/`...)
làm 1 repo cũng được — cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/`
và `scripts/`) ở bất kỳ độ sâu nào trong repo, không cần đúng ngay gốc repo.

**Nhánh (`REPO_BRANCH`) ở cell dưới đã đặt sẵn `compact/compact-gaussian`** — nhánh
này mới có `pipeline/extra/` (`compact_gaussian.py`, `train_compact.py`,
`pick_detail_boxes.py`). Nếu đổi sang nhánh khác (vd `main`) thì Bước 6 sẽ báo lỗi
thiếu file, vì `main` chưa có `pipeline/extra/`.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin.git"
REPO_BRANCH = "compact/compact-gaussian"  # <-- nhánh Compact Gaussian (pipeline/extra/); đổi nhánh khác nếu cần

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 --branch "{REPO_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone
print(f"Đã clone branch: {REPO_BRANCH}")

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

assert (target / "extra" / "compact_gaussian.py").exists() and (target / "extra" / "train_compact.py").exists(), (
    "Thiếu pipeline/extra/compact_gaussian.py hoặc train_compact.py sau khi clone — "
    "kiểm tra lại REPO_BRANCH ở cell trên có đúng 'compact/compact-gaussian' không."
)
print("Đã xác nhận có pipeline/extra/compact_gaussian.py + train_compact.py.")

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA/phase1/{public_set,private_set1}/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA` cũng
được — cell dưới tự dò tìm thư mục `phase1` ở bất kỳ độ sâu nào trong zip).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/1GUflBBz4hrVtkMcHLfLMwUYk4HsJkXFu/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục phase1 ...")

In [ ]:
# Tự dò thư mục "phase1" (chứa public_set/ hoặc private_set1/) ở bất kỳ đâu trong
# zip vừa giải nén, rồi symlink về đúng vị trí mà pipeline/common/scenes.py cần:
#   /kaggle/working/Dataset/VAI_NVS_DATA/phase1
import os
from pathlib import Path

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (public_set/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/phase1" trước "phase1" thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    p = Path(dirpath)
    if p.name == "phase1" and (("public_set" in dirnames) or ("private_set1" in dirnames)):
        found = p
        break

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'phase1' chứa public_set/private_set1 trong zip vừa giải nén.\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — kiểm tra lại cấu trúc zip bạn đã upload lên Google Drive."
    )

print("Tìm thấy:", found)
target_parent = Path("/kaggle/working/Dataset/VAI_NVS_DATA")
target_parent.mkdir(parents=True, exist_ok=True)
target = target_parent / "phase1"
if target.exists() or target.is_symlink():
    target.unlink() if target.is_symlink() else None
if not target.exists():
    os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 13 scene + scene nào có sparse hợp lệ — dataset đầy đủ
# thì kỳ vọng has_valid_provided_sparse=True cho CẢ 13 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại giá trị BTS_DATASET_ROOT ở cell
# trên có trỏ đúng chỗ chứa public_set/private_set1 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá /kaggle/working/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.split:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 (tuỳ chọn) — Sanity-check hệ toạ độ

Script tự chạy lại COLMAP riêng cho `HCM0249` để so với sparse chính thức, xem có
khớp không — không bắt buộc, chỉ 1 lần đối chiếu cho chắc. Bật/tắt bằng `RUN_SANITY_CHECK`.

In [ ]:
RUN_SANITY_CHECK = False  # <-- đổi thành True khi muốn chạy lại bước đối chiếu này

if RUN_SANITY_CHECK:
    !python /kaggle/working/pipeline/scripts/02_validate_frame.py
else:
    print("Bỏ qua sanity-check (RUN_SANITY_CHECK = False).")

## Bước 6 — Train Compact Gaussian + render 1 scene private

**Đổi `SCENE` thành 1 trong 8 tên sau rồi Save Version** (mỗi lần 1 tên, ở version khác nhau):

- `HCM0249`
- `HCM0254`
- `HCM0276`
- `HCM1439`
- `HNI0131`
- `HNI0265`
- `HNI0366`
- `HNI0437`

Chi tiết đầy đủ ghi ra file `pipeline/work/<scene>/03b_train_compact.log`. Xem tiến độ lúc
train đang chạy: mở 1 cell khác gõ `!tail -n 30 /kaggle/working/pipeline/work/<scene>/03b_train_compact.log`.

In [ ]:
SCENE = "HCM0249"  # <-- đổi thành tên scene private muốn train ở version này
!python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE}

### (Tuỳ chọn) Bước 6b — Khai báo vùng chi tiết cao (ăng-ten/RRU/cáp) cần bảo vệ

Compact Gaussian nén (prune) Gaussian dựa trên 1 mask học được — mặc định nén đều trên
toàn cảnh, có thể vô tình cắt bớt cả những Gaussian mảnh ở ăng-ten/RRU/cáp (vốn đã ít
Gaussian, dễ mất chi tiết nếu bị nén thêm). `pipeline/extra/pick_detail_boxes.py` vẽ
sparse point cloud ra 1 file HTML 3D tương tác để tự đọc toạ độ (x, y, z) rồi khai báo
hộp bao quanh cụm ăng-ten/RRU/cáp vào 1 file JSON — đặt đúng tên
`pipeline/work/<scene>/detail_regions.json`, cell train bên dưới tự phát hiện và dùng
(không có file này thì train bình thường, không lỗi gì). Định dạng file JSON xem
docstring đầu `pipeline/extra/compact_gaussian.py` (lớp `DetailRegionSet`). Bỏ qua bước
này nếu chỉ muốn thử Compact Gaussian mặc định trước — với scene private nộp thật, nên
cân nhắc bật để không mất chi tiết ăng-ten/RRU (phần BTC chấm trọng tâm).


In [ ]:
RUN_PICK_DETAIL_BOXES = False  # <-- đổi True nếu muốn xem sparse point cloud 3D để tự chọn hộp bao vùng chi tiết cao

if RUN_PICK_DETAIL_BOXES:
    !pip install -q plotly
    sparse_dir = f"/kaggle/working/pipeline/work/{SCENE}/colmap/dense/sparse/0"
    out_html = f"/kaggle/working/pipeline/work/{SCENE}/detail_boxes_preview.html"
    %cd /kaggle/working/gaussian-splatting
    !python /kaggle/working/pipeline/extra/pick_detail_boxes.py --sparse "{sparse_dir}" --out "{out_html}"
    %cd /kaggle/working
    print(f"Tải {out_html} về máy rồi mở bằng trình duyệt để xem 3D tương tác.")
    print(f"Sau khi đọc được toạ độ, tự tạo file /kaggle/working/pipeline/work/{SCENE}/detail_regions.json "
          "(xem định dạng ở docstring class DetailRegionSet trong pipeline/extra/compact_gaussian.py) "
          "rồi chạy lại cell train bên dưới — nó sẽ tự phát hiện file.")
else:
    print("Bỏ qua chọn vùng chi tiết cao (RUN_PICK_DETAIL_BOXES=False) — train không bảo vệ vùng riêng nào.")

In [ ]:
import os
os.environ["ITERATIONS"] = "30000"
os.environ["LAMBDA_MASK"] = "5e-4"  # trọng số loss mask L_m — tăng để nén mạnh hơn (giảm số Gaussian), giảm để giữ chất lượng
os.environ["MASK_LR"] = "1e-2"
os.environ["MASK_PRUNE_FROM_ITER"] = "1500"
os.environ["MASK_PRUNE_INTERVAL"] = "100"
os.environ["HARD_PROTECT_DETAIL"] = "1"  # "0" -> chỉ giảm trọng số loss mask ở vùng chi tiết cao thay vì bảo vệ cứng (xem Bước 6b)
!bash /kaggle/working/pipeline/scripts/03b_train_compact.sh {SCENE}
!tail -n 6 /kaggle/working/pipeline/work/{SCENE}/03b_train_compact.log

In [ ]:
!python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {SCENE} --model_dir /kaggle/working/pipeline/work/{SCENE}/gs_model_compact

**Không chạy `05_eval_metrics.py` ở đây** — private không có ảnh ground-truth để so
sánh (`has_valid_provided_sparse`/`gt_test_images_dir` chỉ có ở public_set, xem
`pipeline/common/scenes.py`). Cell dưới hiển thị vài ảnh render ra để tự mắt kiểm tra
hợp lý (không nhiễu loạn, không sai màu/hình dạng bất thường) trước khi lưu lên Drive.

In [ ]:
# Xem thử vài ảnh render ra (kiểm tra bằng mắt — không có điểm số vì private không có ảnh thật để so).
from pathlib import Path
from IPython.display import display
from PIL import Image

renders_dir = Path(f"/kaggle/working/pipeline/work/{SCENE}/renders")
all_renders = sorted(renders_dir.glob("*.png"))
sample = all_renders[:4]
print(f"{len(all_renders)} ảnh render tại {renders_dir}, xem thử {len(sample)} ảnh đầu:")
for p in sample:
    display(Image.open(p))

## Bước 7 — Lấy checkpoint để lưu lên Google Drive

**Lưu ý (đã sửa)**: bản hướng dẫn cũ ghi "upload cả thư mục" nhưng
`kaggle_submission.ipynb` Bước 5 thực ra chỉ tải về đúng **1 file `point_cloud.ply`**
mỗi scene (không phải thư mục) — 2 notebook từng lệch nhau ở điểm này. Bên dưới đã sửa
đúng: chỉ cần lấy 1 file `.ply`.

Checkpoint (trọng số đã train, đã nén bằng Compact Gaussian) nằm ở:
`pipeline/work/<SCENE>/gs_model_compact/point_cloud/iteration_30000/point_cloud.ply`
(kèm 2 checkpoint giữa chừng ở `iteration_7000/` và `iteration_15000/`, phòng khi cần
iteration cuối bị lỗi — CHỈ cần lấy đúng file `.ply` ở `iteration_30000/`, không cần lấy
2 checkpoint giữa chừng kia).

Cách lấy: bấm **Save Version**, vào tab **Output**, tìm đúng file
`pipeline/work/<SCENE>/gs_model_compact/point_cloud/iteration_30000/point_cloud.ply`
(không cần lấy nguyên `/kaggle/working` hay cả thư mục `gs_model_compact` — phần còn lại
chỉ là code/dataset/repo clone/checkpoint giữa chừng, không cần cho submission), tải
ĐÚNG FILE `.ply` này về máy rồi upload lên Google Drive dưới dạng **1 file đơn** (KHÔNG
phải cả thư mục) — đặt tên file rõ theo tên scene (vd `<SCENE>_point_cloud.ply`) để
không nhầm lẫn khi điền link ở `kaggle_submission.ipynb`. Nhớ đổi chế độ share file đó
thành "Anyone with the link".

Chạy xong 8 lần (8 version, 8 scene khác nhau) là đủ toàn bộ private_set1.